In [1]:
# The paper claims TW-IDF beats BM25 and TF-IDF. We cannot test that claim until we have BM25 and TF-IDF scores on our own data.


#loading the toolbox, environment and dataset
import sys
sys.path.append("../src")

import pandas as pd

from gow_rag.data_loader import load_corpus, load_claims
from gow_rag.preprocessing import tokenize
from gow_rag.sparse_retrieval import SparseRetriever, count_words
from gow_rag.evaluation import evaluate
from gow_rag.paths import TABLES_DIR

corpus = load_corpus()
queries, qrels = load_claims("validation")

print("Papers:", len(corpus))
print("Claims:", len(queries))

Papers: 5183
Claims: 188


In [2]:
# Turn every paper into word counts
doc_ids = corpus["doc_id"].tolist()
doc_tokens = [tokenize(text) for text in corpus["full_text"]]

doc_counts = [count_words(tokens) for tokens in doc_tokens]   
doc_lengths = [len(tokens) for tokens in doc_tokens]          

print("First paper has", doc_lengths[0], "words")
print("5 frequent words:", sorted(doc_counts[0].items(), key=lambda x: -x[1])[:5])
print("Average paper length:", round(sum(doc_lengths) / len(doc_lengths), 1), "words")

First paper has 202 words
5 frequent words: [('white', 10), ('matter', 10), ('diffusion', 9), ('infants', 6), ('cerebral', 5)]
Average paper length: 147.7 words


In [3]:
#Building the search engines
tfidf = SparseRetriever(doc_ids, doc_counts, doc_lengths, model="tfidf")
bm25 = SparseRetriever(doc_ids, doc_counts, doc_lengths, model="bm25")

print("Search engines ready. Vocabulary size:", len(bm25.vocab))

Search engines ready. Vocabulary size: 46916


In [4]:
#Test run
first_id = list(queries)[0]
print("Claim:", queries[first_id])
print("Correct paper(s):", qrels[first_id])
print()

titles = dict(zip(corpus["doc_id"], corpus["title"]))
top5 = bm25.search(tokenize(queries[first_id]), top_k=5)
for rank, doc_id in enumerate(top5, start=1):
    mark = "<-- CORRECT" if doc_id in qrels[first_id] else ""
    print(rank, doc_id, titles[doc_id][:70], mark)

Claim: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
Correct paper(s): {'14717500'}

1 14717500 Rare Variants Create Synthetic Genome-Wide Associations <-- CORRECT
2 3672261 Natural variation in the parameters of innate immune cells is preferen 
3 2739854 Rare and common variants: twenty arguments 
4 23389795 Common and rare variants in multifactorial susceptibility to common di 
5 4632921 Large-Scale Profiling Reveals the Influence of Genetic Variation on Ge 


In [5]:
#Scoring with dataset
runs = {
    "TF-IDF": tfidf.search_all(queries, tokenize),
    "BM25": bm25.search_all(queries, tokenize),
}

results = pd.DataFrame({name: evaluate(run, qrels) for name, run in runs.items()}).T
results.index.name = "Method"
results

,MAP,P@10,Recall@10,MRR@10,nDCG@10
Method,,,,,
TF-IDF,0.7601,0.0957,0.8902,0.7636,0.7914
BM25,0.7704,0.0979,0.9115,0.7749,0.8049


In [6]:
#saving the table
output_file = TABLES_DIR / "classic_baselines_validation.csv"
results.to_csv(output_file)
print("Saved to:", output_file)

Saved to: C:\Users\Lenovo\Desktop\Projects for IP Paris Application\Graph of words (Reviewing the paper with dataset)\github_upload\scientific-literature-GOW-RAG\results\tables\classic_baselines_validation.csv
